In [8]:
import pathlib
import warnings

import numpy as np
from numpy.lib.recfunctions import stack_arrays
from tqdm import tqdm

from astroquery.gaia import Gaia
import pyia
import gala.coordinates as gc
import astropy.coordinates as coord
import astropy.units as u
import astropy.table as at
from astropy.io import fits

In [54]:
Gaia.login(
    credentials_file=pathlib.Path('~/.gaia/archive.login').expanduser()
)

INFO: Login to gaia TAP server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]
INFO: Login to gaia data server [astroquery.gaia.core]
INFO: OK [astroquery.utils.tap.core]


In [9]:
data_path = pathlib.Path('../../data/gd1-polygon/').resolve()
data_path.mkdir(exist_ok=True, parents=True)

In [56]:
gaia_cols_lite = [
    'lite.source_id', 
    'lite.ra', 'lite.dec', 
    'lite.parallax', 'lite.parallax_error', 
    'lite.pmra', 'lite.pmra_error', 'lite.pmdec', 'lite.pmdec_error',
    'lite.radial_velocity', 'lite.radial_velocity_error',
    'lite.phot_g_mean_mag', 'lite.phot_g_mean_flux_over_error', 
    'lite.phot_bp_mean_mag', 'lite.phot_bp_mean_flux_over_error', 
    'lite.phot_rp_mean_mag', 'lite.phot_rp_mean_flux_over_error', 
    'lite.ruwe', 
]

gaia_cols_full = [
    'full_tbl.ra_parallax_corr', 'full_tbl.ra_pmra_corr', 'full_tbl.ra_pmdec_corr', 'full_tbl.dec_parallax_corr', 
    'full_tbl.dec_pmra_corr', 'full_tbl.dec_pmdec_corr', 'full_tbl.parallax_pmra_corr', 
    'full_tbl.parallax_pmdec_corr', 'full_tbl.pmra_pmdec_corr'
]

In [57]:
q_base = '''
SELECT {cols_lite}, {cols_full}
FROM gaiadr3.gaia_source_lite AS lite
JOIN gaiadr3.gaia_source AS full_tbl
        ON lite.source_id = full_tbl.source_id
WHERE lite.parallax < 1 AND
    CONTAINS(POINT('ICRS', lite.ra, lite.dec), 
            POLYGON('ICRS', 
                    {c[0].ra.degree}, {c[0].dec.degree}, 
                    {c[1].ra.degree}, {c[1].dec.degree}, 
                    {c[2].ra.degree}, {c[2].dec.degree}, 
                    {c[3].ra.degree}, {c[3].dec.degree})) = 1
'''

In [58]:
queries = []
jobs = {}
for i, l in enumerate(np.arange(-100, 20, 10)):
    print(l)
    
    fn = data_path / f'gd1_{l:.0f}.fits'
    if fn.exists():
        print(f'{fn!s} exists...skipping')
        continue

    corners = gc.GD1Koposov10(
        phi1=[l, l, l+10, l+10]*u.deg, 
        phi2=[-10, 5, 5, -10]*u.deg
    )
    corners_icrs = corners.transform_to(coord.ICRS())
    q = q_base.format(
        cols_lite=', '.join(gaia_cols_lite),
        cols_full=', '.join(gaia_cols_full),
        c=corners_icrs
    )
    print(q)
        
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        job = Gaia.launch_job_async(q, name=f'GD1_{i:02d}',
                                    background=True)
    jobs[fn] = job

-100

SELECT lite.source_id, lite.ra, lite.dec, lite.parallax, lite.parallax_error, lite.pmra, lite.pmra_error, lite.pmdec, lite.pmdec_error, lite.radial_velocity, lite.radial_velocity_error, lite.phot_g_mean_mag, lite.phot_g_mean_flux_over_error, lite.phot_bp_mean_mag, lite.phot_bp_mean_flux_over_error, lite.phot_rp_mean_mag, lite.phot_rp_mean_flux_over_error, lite.ruwe, full_tbl.ra_parallax_corr, full_tbl.ra_pmra_corr, full_tbl.ra_pmdec_corr, full_tbl.dec_parallax_corr, full_tbl.dec_pmra_corr, full_tbl.dec_pmdec_corr, full_tbl.parallax_pmra_corr, full_tbl.parallax_pmdec_corr, full_tbl.pmra_pmdec_corr
FROM gaiadr3.gaia_source_lite AS lite
JOIN gaiadr3.gaia_source AS full_tbl
        ON lite.source_id = full_tbl.source_id
WHERE lite.parallax < 1 AND
    CONTAINS(POINT('ICRS', lite.ra, lite.dec), 
            POLYGON('ICRS', 
                    124.90645851283158, -19.955203131285995, 
                    111.39341430156357, -12.408913634684536, 
                    116.59156945722076,

In [59]:
if jobs:
    for fn, job in tqdm(jobs.items(), total=len(jobs)):
        tbl = job.get_results()

#         for c in tbl.colnames: # hack to make sure object arrays are string
#             if tbl[c].dtype == object:
#                 tbl[c] = np.array(tbl[c]).astype(str)
        tbl.write(fn, overwrite=True)

100%|██████████| 12/12 [16:33<00:00, 82.79s/it]  


In [106]:
stream_frame = gc.GD1Koposov10()

all_filename = pathlib.Path('../../gd1-dr3-all.fits')
if not all_filename.exists():
    arrs = []
    for filename in data_path.glob('*.fits'):
        arrs.append(np.array(fits.getdata(filename)))
    arr = stack_arrays(arrs, asrecarray=True, usemask=False)

    tbl = pyia.GaiaData(arr)

    c_gd1 = tbl.get_skycoord(distance=False).transform_to(stream_frame) # uses reference epoch of latest Gaia release by default
    tbl['phi1'] = getattr(c_gd1, "phi1")
    tbl['phi2'] = getattr(c_gd1, "phi2")
    tbl['pm_phi1_cosphi2_unrefl'] = getattr(c_gd1, "pm_phi1_cosphi2")
    tbl['pm_phi2_unrefl'] = getattr(c_gd1, "pm_phi2")

    pm_cov, _ = tbl.get_cov(coords=['pmra', 'pmdec'])
    c = coord.SkyCoord(ra = tbl['ra'], dec=tbl['dec'])
    gd1_coord_pm_cov = gc.transform_pm_cov(c=c, cov=pm_cov, to_frame=stream_frame)

    tbl['pm1_error'] = np.sqrt(gd1_coord_pm_cov[:,0,0]) * u.mas/u.yr
    tbl['pm2_error'] = np.sqrt(gd1_coord_pm_cov[:,1,1]) * u.mas/u.yr
    
    tbl.data.write(all_filename, overwrite=True)

## PS1 Download

In [109]:
tbl = at.Table.read('../../data/gd1-dr3-all.fits')
tbl = tbl[tbl['phi1'] > -100]
len(tbl)

8422003

In [2]:
import mastcasjobs
import requests

import os
import numpy as np
import matplotlib.pyplot as plt
import json

# get the WSID and password if not already defined
import getpass
if not os.environ.get('CASJOBS_USERID'):
    os.environ['CASJOBS_USERID'] = 'ktavangar'
if not os.environ.get('CASJOBS_PW'):
    os.environ['CASJOBS_PW'] = 'Sharksinthecc21!m'

In [61]:
base_ps1_query = """
    select o.objID, o.raMean, o.decMean,
    o.nDetections, o.ng, o.nr, o.ni, o.nz, o.ny,
    m.gMeanPSFMag, m.rMeanPSFMag, m.iMeanPSFMag, m.zMeanPSFMag, m.yMeanPSFMag,
    m.gMeanPSFMagErr, m.rMeanPSFMagErr, m.iMeanPSFMagErr, m.zMeanPSFMagErr, m.yMeanPSFMagErr,
    m.gMeanKronMag, m.rMeanKronMag, m.iMeanKronMag, m.zMeanKronMag, m.yMeanKronMag, 
    m.gMeanKronMagErr, m.rMeanKronMagErr, m.iMeanKronMagErr, m.zMeanKronMagErr, m.yMeanKronMagErr
    INTO MyDB.{batch_name}
    FROM fGetObjFromRectEq({ra_min}, {dec_min}, {ra_max}, {dec_max}) r
    inner join ObjectThin o on o.objid=r.objid and o.nDetections>1
    inner join MeanObject m on o.objid=m.objid and o.uniquePspsOBid=m.uniquePspsOBid
    WHERE m.gMeanPSFMag > -999 AND m.rMeanPSFMag > -999 AND m.iMeanPSFMag > -999
        AND m.gMeanKronMag > -999 AND m.rMeanKronMag > -999 AND m.iMeanKronMag > -999
        AND (m.iMeanPSFMag - m.iMeanKronMag) < 0.05
"""

In [62]:
job_ids = np.array([])
for i, l in enumerate(np.arange(-100, 20, 2)):
    
    fn = data_path / f'gd1_ps1_{l:.0f}.fits'
    if fn.exists():
        print(f'{fn!s} exists...skipping')
        continue

    batch_name = f'GD1_PS1_{i:02d}'

    corners = gc.GD1Koposov10(
        phi1=[l, l, l+10, l+10]*u.deg, 
        phi2=[-10, 5, 5, -10]*u.deg
    )
    corners_icrs = corners.transform_to(coord.ICRS())
    q = base_ps1_query.format(batch_name=batch_name,
        ra_min=corners_icrs.ra.min().value,
        dec_min=corners_icrs.dec.min().value, 
        ra_max=corners_icrs.ra.max().value, 
        dec_max=corners_icrs.dec.max().value)
        
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        job = mastcasjobs.MastCasJobs(context="PanSTARRS_DR2")
        job_id = job.submit(q, task_name=batch_name)
        print(f"Submitted job {job_id}, phi1_min={l:.0f}")
        job_ids = np.append(job_ids, job_id)

Submitted job 118148648, phi1_min=-100
Submitted job 118148649, phi1_min=-98
Submitted job 118148650, phi1_min=-96
Submitted job 118148651, phi1_min=-94
Submitted job 118148652, phi1_min=-92
Submitted job 118148653, phi1_min=-90
Submitted job 118148654, phi1_min=-88
Submitted job 118148655, phi1_min=-86
Submitted job 118148656, phi1_min=-84
Submitted job 118148657, phi1_min=-82
Submitted job 118148658, phi1_min=-80
Submitted job 118148659, phi1_min=-78
Submitted job 118148660, phi1_min=-76
Submitted job 118148661, phi1_min=-74
Submitted job 118148662, phi1_min=-72
Submitted job 118148663, phi1_min=-70
Submitted job 118148664, phi1_min=-68
Submitted job 118148665, phi1_min=-66
Submitted job 118148667, phi1_min=-64
Submitted job 118148668, phi1_min=-62
Submitted job 118148669, phi1_min=-60
Submitted job 118148670, phi1_min=-58
Submitted job 118148671, phi1_min=-56
Submitted job 118148672, phi1_min=-54
Submitted job 118148673, phi1_min=-52
Submitted job 118148674, phi1_min=-50
Submitted j

In [111]:
sub_tbl = tbl['source_id', 'ra', 'dec']
path = data_path / 'tmp'
path.mkdir(exist_ok=True)

idx = np.linspace(0, len(tbl), 4).astype(int)
for j, (i1, i2) in enumerate(zip(idx[:-1], idx[1:])):
    sub_tbl[i1:i2].write(
        path / f'gd1_all_mast_upload{j}.csv', 
        overwrite=True,
        format='ascii.csv'
    )

## Other Radial Velocities

### APOGEE

In [ ]:
allstar = at.Table.read('/mnt/home/apricewhelan/data/APOGEE_DR17/allStarLite-dr17-synspec_rev1.fits')
allstar_c = coord.SkyCoord(
    allstar['RA'][np.isfinite(allstar['RA'])], 
    allstar['DEC'][np.isfinite(allstar['RA'])], 
    unit=u.deg
)

### SDSS

In [ ]:
# get the WSID and password if not already defined
import getpass
if not os.environ.get('CASJOBS_USERID'):
    os.environ['CASJOBS_USERID'] = 'ktavangar'
if not os.environ.get('CASJOBS_PW'):
    os.environ['CASJOBS_PW'] = 'Sharksinthecc21!m'

from astropy.constants import c as speed_of_light

query = """
SELECT
    s.specObjID NUMERIC, s.ra FLOAT, s.dec FLOAT, s.class, s.zWarning INT,
    s.z REAL, s.zErr REAL,
    p.ELODIERVFINAL REAL, p.ELODIERVFINALERR REAL
INTO MyDB.sdss_rv
FROM SpecObjAll AS s
LEFT JOIN sppParams AS p ON s.specObjID = p.specObjID
WHERE
    s.class = 'STAR'
    AND s.zWarning = 0
    AND s.seguePrimary = 1
"""

jobs = mastcasjobs.MastCasJobs(context="SDSSDR17")
results = jobs.submit(query, task_name="sdss_rvs1")

In [ ]:
# Convert pipeline z to velocity in km/s
c_kms = speed_of_light.to(u.km/u.s)
results['v_pipeline'] = results['z_pipeline'] * c_kms
results['v_pipeline_err'] = results['zErr_pipeline'] * c_kms

# Cut to the GD-1 stream track
coords = SkyCoord(ra=results['ra']*u.deg, dec=results['dec']*u.deg)
gd1_coords = coords.transform_to(gc.GD1Koposov10)
phi1, phi2 = gd1_coords.phi1.deg, gd1_coords.phi2.deg
mask = np.abs(phi2) < 2.0
results_stream = results[mask]

n_stars
int64
60228
